# Hands-on Image Retrieval Workshop

In this notebook you will recreate the core search technology used in the prototype.

The goal is simple: given a query image, find the most visually similar objects in a gallery.

We will use:

- **DINOv2** to turn each image into an embedding vector.
- **FAISS** to search quickly through those vectors.
- **Top-k retrieval** to inspect the best candidate objects.

Run the notebook from top to bottom. The code is intentionally straightforward so you can modify it during the workshop.

## Notebook Navigation

1. [Correct Folder Structure](#correct-folder-structure)
2. [Optional Package Installation](#optional-package-installation)
3. [Imports and Paths](#imports-and-paths)
4. [Load Images from the Workshop Folders](#load-images)
5. [Quick Preview](#quick-preview)
6. [Load DINOv2](#load-dinov2)
7. [Embed Images](#embed-images)
8. [Build the FAISS Gallery Index](#build-faiss-index)
9. [Search One Query Image](#search-one-query)
10. [Visualize the Top-3 Results](#visualize-top3)
11. [Search Every Query Image](#search-every-query)
12. [Try It Yourself](#try-it-yourself)

<a id="correct-folder-structure"></a>

## 1. Correct Folder Structure

Before running the search, put your images into this structure:

```text
workshop_materials/
  data/
    gallery/
      object_001/
        img_001.jpg
        img_002.jpg
      object_002/
        img_001.jpg
    query/
      query_001.jpg
      query_002.jpg
    query_labels.csv
```

Rules:

- Each known object gets one folder inside `data/gallery/`.
- Put multiple images of the same object in the same object folder.
- Put query images directly inside `data/query/`.
- Folder names such as `object_001` are the object identifiers.
- `query_labels.csv` is optional, but useful if you want to check accuracy.

Example `query_labels.csv`:

```csv
query_name,true_object
query_001.jpg,object_001
query_002.jpg,object_002
query_unknown.jpg,
```

<a id="optional-package-installation"></a>

## 2. Optional Package Installation

If this notebook runs in a fresh environment, uncomment and run the next cell once.

If the packages are already installed, skip it.

In [ ]:
# Uncomment if needed:
# %pip install -q pillow numpy pandas matplotlib torch transformers faiss-cpu tqdm

<a id="imports-and-paths"></a>

## 3. Imports and Paths

This cell finds the `workshop_materials/` folder automatically, so the notebook works whether you open it from the notebook folder or from the repository root.

In [ ]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm


def find_workshop_dir() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        possible_locations = [
            candidate,
            candidate / "workshop_materials",
            candidate / "step_1" / "workshop_materials",
        ]
        for location in possible_locations:
            if (location / "data").exists() and (location / "notebooks").exists():
                return location.resolve()
    raise FileNotFoundError("Could not find workshop_materials/. Run this notebook from inside the repository.")


WORKSHOP_DIR = find_workshop_dir()
DATA_DIR = WORKSHOP_DIR / "data"
GALLERY_DIR = DATA_DIR / "gallery"
QUERY_DIR = DATA_DIR / "query"
LABELS_PATH = DATA_DIR / "query_labels.csv"
OUTPUTS_DIR = WORKSHOP_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Workshop folder:", WORKSHOP_DIR)
print("Gallery folder:", GALLERY_DIR)
print("Query folder:", QUERY_DIR)
print("Outputs folder:", OUTPUTS_DIR)

<a id="load-images"></a>

## 4. Load Images from the Workshop Folders

The gallery is the known collection. The query folder contains images we want to search for.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTENSIONS


def list_images(folder: Path) -> list[Path]:
    if not folder.exists():
        return []
    return sorted(path for path in folder.rglob("*") if path.is_file() and is_image_file(path))


def load_gallery_dataframe(gallery_dir: Path) -> pd.DataFrame:
    rows = []
    for object_dir in sorted(gallery_dir.iterdir() if gallery_dir.exists() else []):
        if not object_dir.is_dir():
            continue
        object_id = object_dir.name
        for image_path in list_images(object_dir):
            rows.append({
                "object_id": object_id,
                "image_name": image_path.name,
                "image_path": str(image_path.resolve()),
                "image_relpath": str(image_path.relative_to(gallery_dir)),
            })
    return pd.DataFrame(rows)


gallery_df = load_gallery_dataframe(GALLERY_DIR)
query_paths = list_images(QUERY_DIR)

if gallery_df.empty:
    raise ValueError("No gallery images found. Add images under data/gallery/object_001/, object_002/, etc.")

if not query_paths:
    raise ValueError("No query images found. Add images directly under data/query/.")

print(f"Gallery images: {len(gallery_df)}")
print(f"Gallery objects: {gallery_df['object_id'].nunique()}")
print(f"Query images: {len(query_paths)}")

display(gallery_df.groupby("object_id").size().rename("image_count").reset_index())

<a id="quick-preview"></a>

## 5. Quick Preview

This preview is just to check that the notebook is reading the images you expected.

In [ ]:
def load_rgb(path: Path) -> Image.Image:
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        return image.convert("RGB")


def show_image_grid(paths: list[Path], titles: list[str] | None = None, max_images: int = 8):
    paths = paths[:max_images]
    if not paths:
        print("No images to show.")
        return
    cols = min(4, len(paths))
    rows = int(np.ceil(len(paths) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for i, path in enumerate(paths):
        axes[i].imshow(load_rgb(path))
        axes[i].set_title(titles[i] if titles else path.name, fontsize=10)
    plt.tight_layout()
    plt.show()


sample_gallery_paths = [Path(path) for path in gallery_df["image_path"].head(8)]
sample_gallery_titles = [f"{row.object_id}\n{row.image_name}" for row in gallery_df.head(8).itertuples()]

print("Gallery preview")
show_image_grid(sample_gallery_paths, sample_gallery_titles)

print("Query preview")
show_image_grid(query_paths, [path.name for path in query_paths])

<a id="load-dinov2"></a>

## 6. Load DINOv2

DINOv2 converts each image into a vector. A vector is just a list of numbers, but it carries visual information about the image.

Images that look similar should have vectors that are close together.

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel

MODEL_NAME = "facebook/dinov2-base"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

print("Loaded model:", MODEL_NAME)
print("Device:", DEVICE)

<a id="embed-images"></a>

## 7. Embed Images

This function turns image files into normalized vectors. Normalization makes cosine similarity easy to compute.

In [ ]:
def l2_normalize(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.maximum(norms, 1e-12)


@torch.inference_mode()
def embed_images(image_paths: list[Path], batch_size: int = 8) -> np.ndarray:
    all_embeddings = []
    for start in tqdm(range(0, len(image_paths), batch_size), desc="Embedding images"):
        batch_paths = image_paths[start:start + batch_size]
        images = [load_rgb(path) for path in batch_paths]
        inputs = processor(images=images, return_tensors="pt")
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

        outputs = model(**inputs)
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            embeddings = outputs.pooler_output
        else:
            embeddings = outputs.last_hidden_state.mean(dim=1)

        all_embeddings.append(embeddings.detach().cpu().float().numpy())

    embeddings = np.concatenate(all_embeddings, axis=0).astype("float32")
    return l2_normalize(embeddings)

<a id="build-faiss-index"></a>

## 8. Build the FAISS Gallery Index

FAISS stores the gallery vectors and lets us search for the nearest ones quickly.

Because our vectors are normalized, inner product search is equivalent to cosine similarity.

In [ ]:
import faiss

gallery_paths = [Path(path) for path in gallery_df["image_path"]]
gallery_embeddings = embed_images(gallery_paths, batch_size=8)

embedding_dim = gallery_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(gallery_embeddings)

np.save(OUTPUTS_DIR / "gallery_embeddings.npy", gallery_embeddings)
faiss.write_index(index, str(OUTPUTS_DIR / "gallery_faiss.index"))
gallery_df[["object_id", "image_relpath", "image_name"]].to_csv(OUTPUTS_DIR / "gallery_metadata.csv", index=False)

print("Gallery index built.")
print("Embedding shape:", gallery_embeddings.shape)
print("Saved outputs to:", OUTPUTS_DIR)

<a id="search-one-query"></a>

## 9. Search One Query Image

Now we embed one query image and ask FAISS for the nearest gallery images.

Then we group the image matches by object folder, so the result is object-level rather than just image-level.

In [ ]:
def object_results_from_image_matches(scores: np.ndarray, indices: np.ndarray, top_k_objects: int = 5) -> list[dict]:
    best_by_object = {}
    for score, idx in zip(scores, indices):
        if idx < 0:
            continue
        row = gallery_df.iloc[int(idx)]
        object_id = row["object_id"]
        match = {
            "object_id": object_id,
            "score": float(score),
            "image_name": row["image_name"],
            "image_path": row["image_path"],
        }
        if object_id not in best_by_object or match["score"] > best_by_object[object_id]["score"]:
            best_by_object[object_id] = match

    return sorted(best_by_object.values(), key=lambda item: item["score"], reverse=True)[:top_k_objects]


def search_query(query_path: Path, top_k_images: int = 30, top_k_objects: int = 5) -> list[dict]:
    query_embedding = embed_images([query_path], batch_size=1)
    search_count = min(top_k_images, len(gallery_df))
    scores, indices = index.search(query_embedding, search_count)
    return object_results_from_image_matches(scores[0], indices[0], top_k_objects=top_k_objects)


selected_query = query_paths[0]
single_query_results = search_query(selected_query, top_k_images=30, top_k_objects=5)

print("Selected query:", selected_query.name)
display(pd.DataFrame(single_query_results))

<a id="visualize-top3"></a>

## 10. Visualize the Top-3 Results

This is the most important workshop view: query image on the left, model picks on the right.

In [ ]:
PALETTE = {
    "blue": "#071951",
    "coral": "#ff5757",
    "orange": "#f9b22f",
    "green": "#1c8b5a",
}


def lookup_true_object(query_name: str) -> str | None:
    if not LABELS_PATH.exists():
        return None
    labels_df = pd.read_csv(LABELS_PATH)
    if labels_df.empty or "query_name" not in labels_df.columns or "true_object" not in labels_df.columns:
        return None
    match = labels_df[labels_df["query_name"].astype(str).eq(query_name)]
    if match.empty:
        return None
    value = match.iloc[0]["true_object"]
    if pd.isna(value) or str(value).strip() == "":
        return None
    return str(value).strip()


def visualize_query_results(query_path: Path, results: list[dict], true_object: str | None = None, top_k: int = 3):
    shown_results = results[:top_k]
    fig, axes = plt.subplots(1, len(shown_results) + 1, figsize=(4.5 * (len(shown_results) + 1), 4.8))
    axes = np.array(axes).reshape(-1)

    axes[0].imshow(load_rgb(query_path))
    axes[0].set_title(f"Query\n{query_path.name}", color=PALETTE["blue"], fontweight="bold")
    axes[0].axis("off")

    for rank, result in enumerate(shown_results, start=1):
        ax = axes[rank]
        object_id = result["object_id"]
        is_correct = true_object is not None and object_id == true_object
        title_color = PALETTE["green"] if is_correct else (PALETTE["coral"] if rank == 1 else PALETTE["blue"])
        ax.imshow(load_rgb(Path(result["image_path"])))
        ax.set_title(
            f"Top {rank}: {object_id}\nscore={result['score']:.3f}\n{result['image_name']}",
            color=title_color,
            fontweight="bold" if rank == 1 or is_correct else "normal",
        )
        ax.axis("off")

    if true_object:
        fig.suptitle(f"Known truth: {true_object}", color=PALETTE["blue"], fontsize=14, fontweight="bold")
    else:
        fig.suptitle("No known truth label for this query", color=PALETTE["blue"], fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.show()


true_object = lookup_true_object(selected_query.name)
visualize_query_results(selected_query, single_query_results, true_object=true_object, top_k=3)

<a id="search-every-query"></a>

## 11. Search Every Query Image

This runs the same search for every image in `data/query/` and saves a simple results table.

In [ ]:
all_rows = []

for query_path in tqdm(query_paths, desc="Searching queries"):
    results = search_query(query_path, top_k_images=30, top_k_objects=5)
    ranked_ids = [result["object_id"] for result in results]
    true_object = lookup_true_object(query_path.name)

    all_rows.append({
        "query_name": query_path.name,
        "true_object": true_object,
        "top1_object": ranked_ids[0] if ranked_ids else None,
        "top1_score": results[0]["score"] if results else None,
        "top3_objects": ranked_ids[:3],
        "top1_correct": bool(true_object and ranked_ids and ranked_ids[0] == true_object),
        "top3_correct": bool(true_object and true_object in ranked_ids[:3]),
    })

results_df = pd.DataFrame(all_rows)
results_df.to_csv(OUTPUTS_DIR / "query_results.csv", index=False)

display(results_df)
print("Saved:", OUTPUTS_DIR / "query_results.csv")

<a id="try-it-yourself"></a>

## 12. Try It Yourself

Workshop exercise:

1. Add a new object folder inside `data/gallery/`, for example `object_003/`.
2. Put two or more images of that object into the folder.
3. Add one or more query images into `data/query/`.
4. If you know the correct answer, add a row to `data/query_labels.csv`.
5. Rerun the notebook from **Load Images from the Workshop Folders** onward.

Things to test:

- Does the same object work from a different angle?
- What happens if the background changes?
- What happens with a visually similar but different object?
- Does Top-3 help when Top-1 is wrong?

This is the core idea of the prototype: embeddings do not give a perfect answer, but they can quickly produce useful visual candidates for human inspection.